# Automated Web Security Header Scanner

**Project:** IBM SkillBuild National Cybersecurity Bootcamp

This notebook checks one authorized website for HTTPS usage and six common HTTP security headers. It is an educational defensive-security tool, not a full penetration test.

## Ethical use
Only run scans against websites you own or have explicit permission to assess. This notebook sends a single HTTP GET request per scan. It does not exploit vulnerabilities, brute-force, crawl, or modify the target.

## How to use
1. Install dependencies using `pip install -r requirements.txt`.
2. Set `TARGET_URL` in the configuration cell below to an authorized target.
3. Run the cells from top to bottom.
4. Review the findings and optionally save the report as JSON.


## 1. Import libraries
The notebook uses `requests` for the HTTP request and Python's built-in `json` and `datetime` modules for reporting.

In [7]:
import json
from datetime import datetime, timezone
from urllib.parse import urlparse

import requests


## 2. Configure the scan
Replace the example URL with a website you own or are explicitly authorized to test. `example.com` is only a placeholder and should not be represented as your actual project test.

In [18]:
TARGET_URL = "https://google.com"  # Replace with your authorized test URL
TIMEOUT_SECONDS = 8
SAVE_JSON = True
JSON_OUTPUT_FILE = "report.json"


## 3. Define the security headers
The scanner checks for the presence of these headers. Presence alone does not prove that a header is correctly configured.

In [19]:
SECURITY_HEADERS = {
    "Strict-Transport-Security": "Helps browsers enforce HTTPS on future connections.",
    "Content-Security-Policy": "Helps control the sources and behavior of page content.",
    "X-Content-Type-Options": "The nosniff value helps prevent MIME-type sniffing.",
    "X-Frame-Options": "Helps control whether a page can be embedded in frames.",
    "Referrer-Policy": "Controls referrer information sent with navigation requests.",
    "Permissions-Policy": "Restricts selected browser features."
}


## 4. Validate the URL and scan
The request uses a timeout, follows redirects, and keeps TLS certificate verification enabled.

In [20]:
def normalize_url(raw_url):
    raw_url = raw_url.strip()
    if not raw_url.startswith(("http://", "https://")):
        raw_url = "https://" + raw_url
    parsed = urlparse(raw_url)
    if parsed.scheme not in ("http", "https") or not parsed.hostname:
        raise ValueError("Enter a valid HTTP or HTTPS URL.")
    if parsed.username or parsed.password:
        raise ValueError("URLs containing embedded credentials are not accepted.")
    return raw_url

def scan_website(url, timeout=8):
    response = requests.get(
        url,
        timeout=timeout,
        allow_redirects=True,
        headers={"User-Agent": "Educational-Web-Security-Header-Scanner/1.0"},
    )
    response_headers = {k.lower(): v for k, v in response.headers.items()}
    findings = []
    for header, explanation in SECURITY_HEADERS.items():
        value = response_headers.get(header.lower())
        findings.append({
            "header": header,
            "present": value is not None,
            "value": value or "",
            "recommendation": explanation,
        })
    final_scheme = urlparse(response.url).scheme.lower()
    return {
        "scan_time_utc": datetime.now(timezone.utc).isoformat(),
        "requested_url": url,
        "final_url": response.url,
        "status_code": response.status_code,
        "https": final_scheme == "https",
        "server_header_disclosed": "server" in response_headers,
        "headers": findings,
        "note": "This is a basic header review, not proof that a website is secure."
    }


## 5. Run the scan
Run this cell after setting `TARGET_URL`. Network availability and the target's configuration affect the results.

In [21]:
try:
    if not 1 <= TIMEOUT_SECONDS <= 30:
        raise ValueError("TIMEOUT_SECONDS must be between 1 and 30.")
    url = normalize_url(TARGET_URL)
    scan_report = scan_website(url, TIMEOUT_SECONDS)
    print("AUTOMATED WEB SECURITY HEADER SCAN")
    print("=" * 38)
    print("Requested URL:", scan_report["requested_url"])
    print("Final URL:", scan_report["final_url"])
    print("HTTP status:", scan_report["status_code"])
    print("HTTPS:", "Yes" if scan_report["https"] else "No")
    print("Server header disclosed:", scan_report["server_header_disclosed"])
    print("\nSecurity headers:")
    for item in scan_report["headers"]:
        status = "PRESENT" if item["present"] else "MISSING"
        print(f"- {item['header']}: {status}")
        if item["present"]:
            print("    Value:", item["value"])
        else:
            print("    Recommendation:", item["recommendation"])
    print("\nNote:", scan_report["note"])
except requests.exceptions.SSLError:
    print("TLS/SSL validation failed. Certificate verification was not disabled.")
except requests.exceptions.Timeout:
    print("The server did not respond before the timeout.")
except requests.exceptions.RequestException as exc:
    print("Network/request error:", exc)
except ValueError as exc:
    print("Input/configuration error:", exc)


AUTOMATED WEB SECURITY HEADER SCAN
Requested URL: https://google.com
Final URL: https://www.google.com/
HTTP status: 200
HTTPS: Yes
Server header disclosed: True

Security headers:
- Strict-Transport-Security: MISSING
    Recommendation: Helps browsers enforce HTTPS on future connections.
- Content-Security-Policy: MISSING
    Recommendation: Helps control the sources and behavior of page content.
- X-Content-Type-Options: MISSING
    Recommendation: The nosniff value helps prevent MIME-type sniffing.
- X-Frame-Options: PRESENT
    Value: SAMEORIGIN
- Referrer-Policy: MISSING
    Recommendation: Controls referrer information sent with navigation requests.
- Permissions-Policy: MISSING
    Recommendation: Restricts selected browser features.

Note: This is a basic header review, not proof that a website is secure.


## 6. Save results to JSON
This cell saves the real output from the scan. Do not report sample or placeholder values as if they were actual findings.

In [23]:
if SAVE_JSON and 'scan_report' in globals():
    with open(JSON_OUTPUT_FILE, 'w', encoding='utf-8') as output_file:
        json.dump(scan_report, output_file, indent=2, ensure_ascii=False)
    print(f"Report saved to {JSON_OUTPUT_FILE}")
else:
    print("No scan report available to save. Run the scan cell successfully first.")


Report saved to report.json


## 7. Test plan

| Test | Action | Expected outcome |
|---|---|---|
| 1 | Scan an authorized, reachable HTTPS website | Status and header findings are displayed |
| 2 | Enable JSON output | A JSON file is created after a successful scan |
| 3 | Enter malformed URL text | URL validation reports an error |
| 4 | Set timeout to 60 seconds | Configuration validation rejects the value |
| 5 | Use a target that times out or has TLS errors | A clear error is displayed; TLS checks remain enabled |

Record actual outcomes after running these tests. A test should be marked as passed only when the observed result matches the expected result.

## 8. Limitations
- Checks header presence, not every aspect of header quality.
- Does not crawl pages or test authentication, application logic, or server-side vulnerabilities.
- A missing header is not by itself proof of a vulnerability.
- Results depend on the selected URL, redirects, and current server configuration.

## References
- MDN Web Docs — HTTP headers: https://developer.mozilla.org/en-US/docs/Web/HTTP/Headers
- OWASP Secure Headers Project: https://owasp.org/www-project-secure-headers/
- Requests documentation: https://requests.readthedocs.io/
